In [0]:
%sql
DESCRIBE HISTORY nyc_taxi.silver.yellow_trips LIMIT 5;

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE = "nyc_taxi.silver.yellow_trips"
KEY = ["vendor_id", "pickup_datetime", "dropoff_datetime",
       "pickup_location_id", "dropoff_location_id"]

# 4.1 / 4.1b 的分析基于 4.2 改写之前的 Silver，固定版本以保证可复现
PRE_DEDUP_VERSION = 4   # 换成第 1 步查到的版本号
silver = spark.sql(f"SELECT * FROM nyc_taxi.silver.yellow_trips VERSION AS OF {PRE_DEDUP_VERSION}")

total_rows    = silver.count()
distinct_keys = silver.select(KEY).distinct().count()

print(f"total rows    : {total_rows:,}")
print(f"distinct keys : {distinct_keys:,}")
print(f"extra rows    : {total_rows - distinct_keys:,}   (rows that share a key with another row)")

In [0]:
print(silver.count())                 # 应为 21,965,805
print("trip_id" in silver.columns)    # 应为 False

In [0]:
dup_keys = silver.groupBy(KEY).count().filter("count > 1")

display(dup_keys.groupBy("count").agg(F.count("*").alias("num_keys")).orderBy("count"))
display(dup_keys.orderBy(F.desc("count")).limit(10))

In [0]:
business_cols = [c for c in silver.columns if not c.startswith("_")]

exact_distinct = silver.select(business_cols).distinct().count()
print(f"rows             : {total_rows:,}")
print(f"distinct (all business columns): {exact_distinct:,}")
print(f"exact duplicates : {total_rows - exact_distinct:,}")

In [0]:
sample_key = dup_keys.orderBy(F.desc("count")).first()
cond = [F.col(k) == sample_key[k] for k in KEY]
display(silver.filter(cond[0] & cond[1] & cond[2] & cond[3] & cond[4]))

In [0]:
# 1) 取出所有重复键对应的行
dup_rows = silver.join(dup_keys.select(KEY), on=KEY, how="inner")

# 2) 要比较的列：业务列中除了键本身，再加上 _source_file 看两行是否来自不同文件
compare_cols = [c for c in business_cols if c not in KEY] + ["_source_file"]

def nd(c):
    # NULL 安全的"不同值个数"
    return F.count_distinct(F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))).alias(c)

per_group = dup_rows.groupBy(KEY).agg(*[nd(c) for c in compare_cols])

# 3) 每列在多少组里出现差异（>1 个不同值）
summary = per_group.select([F.sum((F.col(c) > 1).cast("int")).alias(c) for c in compare_cols])
display(summary)


In [0]:
show_cols = KEY + ["passenger_count", "payment_type", "fare_amount", "extra", "mta_tax",
                   "tip_amount", "tolls_amount", "total_amount", "trip_distance", "_source_file"]
display(dup_rows.select(show_cols).orderBy(KEY).limit(20))


In [0]:
%sql
ALTER TABLE nyc_taxi.silver.yellow_trips
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

DESCRIBE HISTORY nyc_taxi.silver.yellow_trips LIMIT 3;

In [0]:
from pyspark.sql import functions as F

SILVER_TABLE  = "nyc_taxi.silver.yellow_trips"
CHANGES_TABLE = "nyc_taxi.bronze.yellow_trip_changes"
KEY = ["vendor_id", "pickup_datetime", "dropoff_datetime",
       "pickup_location_id", "dropoff_location_id"]

def make_trip_id(df):
    # 必须和 03_silver 里的公式完全一致
    return df.withColumn("trip_id", F.sha2(F.concat_ws("||",
        *[F.coalesce(F.col(c).cast("string"), F.lit("<NULL>")) for c in KEY + ["trip_seq"]]), 256))

silver = spark.table(SILVER_TABLE)
base = silver.filter("trip_seq = 1 AND hour(pickup_datetime) BETWEEN 10 AND 20")
TS = lambda s: F.lit(s).cast("timestamp")

# UPDATE：5 条计价更正
upd = (base.filter("pickup_date = '2019-03-15'").orderBy("trip_id").limit(5)
       .withColumn("fare_amount",  F.col("fare_amount") + 1)
       .withColumn("total_amount", F.col("total_amount") + 1)
       .withColumn("_change_type", F.lit("UPDATE"))
       .withColumn("_change_ts",   TS("2019-04-02 10:00:00")))

# 同一行程的第二次更正，时间更晚，应该胜出
upd2 = (upd.orderBy("trip_id").limit(1)
        .withColumn("fare_amount",  F.col("fare_amount") + 1)
        .withColumn("total_amount", F.col("total_amount") + 1)
        .withColumn("_change_ts",   TS("2019-04-02 12:00:00")))

# DELETE：3 条作废
dele = (base.filter("pickup_date = '2019-03-16'").orderBy("trip_id").limit(3)
        .withColumn("_change_type", F.lit("DELETE"))
        .withColumn("_change_ts",   TS("2019-04-02 11:00:00")))

# INSERT：4 条迟到的新行程（时间平移 3 秒，生成新的键）
ins = (base.filter("pickup_date = '2019-03-17'").orderBy("trip_id").limit(4)
       .withColumn("pickup_datetime",  F.col("pickup_datetime")  + F.expr("INTERVAL 3 SECONDS"))
       .withColumn("dropoff_datetime", F.col("dropoff_datetime") + F.expr("INTERVAL 3 SECONDS")))
ins = (make_trip_id(ins)
       .withColumn("_change_type", F.lit("INSERT"))
       .withColumn("_change_ts",   TS("2019-04-02 09:00:00")))

changes = (upd.unionByName(upd2).unionByName(dele).unionByName(ins)
           .withColumn("_ingested_at", F.current_timestamp()))

(changes.write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(CHANGES_TABLE))

In [0]:
ch = spark.table(CHANGES_TABLE)
display(ch.groupBy("_change_type").count())

# INSERT 的 trip_id 必须不在 Silver 里；UPDATE / DELETE 的必须在
ids = silver.select("trip_id")
print("INSERT 已存在于 silver:",
      ch.filter("_change_type='INSERT'").join(ids, "trip_id").count(), "(应为 0)")
print("UPDATE/DELETE 能匹配到 silver:",
      ch.filter("_change_type!='INSERT'").select("trip_id").distinct().join(ids, "trip_id").count(), "(应为 8)")

display(ch.select("trip_id", "_change_type", "_change_ts", "fare_amount", "total_amount")
          .orderBy("trip_id", "_change_ts"))

In [0]:
from pyspark.sql import functions as F, Window

TARGET        = "nyc_taxi.silver.yellow_trips"
CHANGES_TABLE = "nyc_taxi.bronze.yellow_trip_changes"

target_cols = spark.table(TARGET).columns
ch = spark.table(CHANGES_TABLE)

def prep(df):
    # 只保留目标表的列和两个控制列；血缘列改为指向变更表
    return (df.withColumn("_source_file", F.lit(CHANGES_TABLE))
              .select(*target_cols, "_change_type", "_change_ts"))

non_key     = [c for c in target_cols if c != "trip_id"]
set_clause  = ",\n  ".join(f"t.`{c}` = s.`{c}`" for c in non_key)
insert_cols = ", ".join(f"`{c}`" for c in target_cols)
insert_vals = ", ".join(f"s.`{c}`" for c in target_cols)

def merge_sql(view):
    return f"""
MERGE INTO {TARGET} AS t
USING {view} AS s
ON t.trip_id = s.trip_id
WHEN MATCHED AND s._change_type = 'DELETE' THEN DELETE
WHEN MATCHED THEN UPDATE SET
  {set_clause}
WHEN NOT MATCHED AND s._change_type != 'DELETE' THEN
  INSERT ({insert_cols}) VALUES ({insert_vals})
"""

print(merge_sql("changes_latest")[:600])   # 先看一眼生成的 SQL 开头

In [0]:
prep(ch).createOrReplaceTempView("changes_raw")
try:
    spark.sql(merge_sql("changes_raw"))
except Exception as e:
    print(type(e).__name__)
    print(str(e)[:400])

In [0]:
%sql
DESCRIBE HISTORY nyc_taxi.silver.yellow_trips LIMIT 2;

In [0]:
w = Window.partitionBy("trip_id").orderBy(F.col("_change_ts").desc())
src_latest = (prep(ch)
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn"))

print("去重后源行数:", src_latest.count(), "(应为 12)")
src_latest.createOrReplaceTempView("changes_latest")

display(spark.sql(merge_sql("changes_latest")))

In [0]:
# 4.5 准备：只定义变量，不修改任何数据。会话重置后从这里往下跑即可
from pyspark.sql import functions as F, Window

TARGET        = "nyc_taxi.silver.yellow_trips"
CHANGES_TABLE = "nyc_taxi.bronze.yellow_trip_changes"
PRE_MERGE_VERSION = 7    # 开启 CDF 的版本，也是 MERGE 之前的状态
MERGE_VERSION     = 8    # 本次 MERGE 产生的版本

# 重新得到去重后的变更源（只读，不执行 MERGE）
w = Window.partitionBy("trip_id").orderBy(F.col("_change_ts").desc())
src_latest = (spark.table(CHANGES_TABLE)
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1")
    .drop("_rn"))
print("src_latest 行数:", src_latest.count(), "(应为 12)")

In [0]:
h = spark.sql(f"DESCRIBE HISTORY {TARGET}").filter(f"version = {MERGE_VERSION}")
display(h.select("version", "operation",
                 F.explode("operationMetrics").alias("metric", "value"))
         .orderBy("metric"))

In [0]:
s = spark.table(TARGET)
total = s.count()
uniq  = s.select("trip_id").distinct().count()
print(f"总行数 {total:,}（应为 21,965,804 − 3 + 4 = 21,965,805）")
print(f"trip_id 唯一数 {uniq:,}")
assert total == uniq, "MERGE 后 trip_id 不唯一！"

In [0]:
ids = [r.trip_id for r in src_latest.select("trip_id").collect()]   # 12 个，数量很少，可以 collect

before = (spark.sql(f"SELECT * FROM {TARGET} VERSION AS OF 7")
          .filter(F.col("trip_id").isin(ids))
          .select("trip_id",
                  F.col("fare_amount").alias("fare_before"),
                  F.col("total_amount").alias("total_before")))
after  = (spark.table(TARGET)
          .filter(F.col("trip_id").isin(ids))
          .select("trip_id",
                  F.col("fare_amount").alias("fare_after"),
                  F.col("total_amount").alias("total_after"),
                  "_source_file"))

cmp = (before.join(after, "trip_id", "full_outer")
             .join(src_latest.select("trip_id", "_change_type", "_change_ts"), "trip_id"))
display(cmp.orderBy("_change_type", "trip_id"))

In [0]:
%sql
SELECT _change_type, _commit_version, count(*) AS n
FROM table_changes('nyc_taxi.silver.yellow_trips', 8)
GROUP BY _change_type, _commit_version
ORDER BY _change_type;

In [0]:
%sql
SELECT trip_id, _change_type, fare_amount, total_amount, _commit_version, _commit_timestamp
FROM table_changes('nyc_taxi.silver.yellow_trips', 8)
WHERE _change_type LIKE 'update%'
ORDER BY trip_id, _change_type DESC;

In [0]:
from pyspark.sql import functions as F, Window

TARGET        = "nyc_taxi.silver.yellow_trips"
CHANGES_TABLE = "nyc_taxi.bronze.yellow_trip_changes"

target_cols = spark.table(TARGET).columns
w = Window.partitionBy("trip_id").orderBy(F.col("_change_ts").desc())
src_latest = (spark.table(CHANGES_TABLE)
    .withColumn("_source_file", F.lit(CHANGES_TABLE))
    .select(*target_cols, "_change_type", "_change_ts")
    .withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn"))
src_latest.createOrReplaceTempView("changes_latest")

non_key     = [c for c in target_cols if c != "trip_id"]
set_clause  = ",\n  ".join(f"t.`{c}` = s.`{c}`" for c in non_key)
insert_cols = ", ".join(f"`{c}`" for c in target_cols)
insert_vals = ", ".join(f"s.`{c}`" for c in target_cols)

# 变化检测：任一业务列不同（null 安全），血缘列不参与比较
compare_cols = [c for c in non_key if not c.startswith("_")]
CHANGED = " OR ".join(f"NOT (t.`{c}` <=> s.`{c}`)" for c in compare_cols)

def merge_sql(view, matched_cond=None):
    extra = f" AND ({matched_cond})" if matched_cond else ""
    return f"""
MERGE INTO {TARGET} AS t
USING {view} AS s
ON t.trip_id = s.trip_id
WHEN MATCHED AND s._change_type = 'DELETE' THEN DELETE
WHEN MATCHED{extra} THEN UPDATE SET
  {set_clause}
WHEN NOT MATCHED AND s._change_type != 'DELETE' THEN
  INSERT ({insert_cols}) VALUES ({insert_vals})
"""

In [0]:
display(spark.sql(merge_sql("changes_latest")))

In [0]:
v8 = spark.sql(f"SELECT * FROM {TARGET} VERSION AS OF 8")
v9 = spark.sql(f"SELECT * FROM {TARGET} VERSION AS OF 9")
print("v8 有而 v9 没有:", v8.exceptAll(v9).count())
print("v9 有而 v8 没有:", v9.exceptAll(v8).count())
print("当前总行数:", spark.table(TARGET).count())

In [0]:
%sql
SELECT _change_type, count(*) AS n
FROM table_changes('nyc_taxi.silver.yellow_trips', 9, 9)
GROUP BY _change_type;

In [0]:
display(spark.sql(merge_sql("changes_latest", CHANGED)))

In [0]:
%sql
DESCRIBE HISTORY nyc_taxi.silver.yellow_trips LIMIT 3;